# Add environmental features to parcels

This reusable feature-engineering notebook adds mean DEM elevation and nearest-lake context to the parcel GeoParquet. Each replacement is atomic, and more environmental feature steps can be added here later.

## Setup

In [ ]:
from pathlib import Path
import os
import shutil
from shared.io import read_data, write_data, file_exists
from data_preparation.features import (
    append_nearest_lake_features,
    append_parcel_elevation,
    append_soil_feature,
    create_dem_mosaic,
    find_dem_tiles,
    validate_projected_metric_crs,
)

In [2]:
import os
import shared.postgress as db_l
db_handler = db_l.create_db_handler(db_name=os.getenv("DB_NAME"))
df_users = db_handler.get_df_from_db(table_name=os.getenv("TBL_USERS"))
display(df_users)

users_list = list(df_users[["username", "password"]].itertuples(index=False, name=None))
users_list

,user_id,username,password
0,1,esagovhub1@gmail.com,A9x!rT3#kLm2
1,2,esagovhub2@gmail.com,M4#tNp9!vXe2
2,3,esagovhub3@gmail.com,F4#tEp4!vXe5
3,4,esagovhub4@gmail.com,B4x!yS9#kLb0
4,5,katrinporo@gmail.com,GL9hbX?=!Dw4pD
5,6,louvarieleni@gmail.com,RZbZ8/p3Ctvhue@
6,7,farfarasmpoytos@gmail.com,1AK_E+!Mf9JvJNz
7,8,sofiaargirou03@gmail.com,arg5Rew97!2U3


[('esagovhub1@gmail.com', 'A9x!rT3#kLm2'),
 ('esagovhub2@gmail.com', 'M4#tNp9!vXe2'),
 ('esagovhub3@gmail.com', 'F4#tEp4!vXe5'),
 ('esagovhub4@gmail.com', 'B4x!yS9#kLb0'),
 ('katrinporo@gmail.com', 'GL9hbX?=!Dw4pD'),
 ('louvarieleni@gmail.com', 'RZbZ8/p3Ctvhue@'),
 ('farfarasmpoytos@gmail.com', '1AK_E+!Mf9JvJNz'),
 ('sofiaargirou03@gmail.com', 'arg5Rew97!2U3')]

## Paths and options

In [ ]:
WORK_PATH = Path("C:/work_dir/neuro_2024-2025")
DEM_DIR = WORK_PATH / "dem"
DEM_DIR.mkdir(parents=True, exist_ok=True)
MOSAIC_PATH = DEM_DIR / "dem_mosaic.tif"

PARCELS_PATH = WORK_PATH / "1_eo_parcel_stats_fill" / "satellite_parcel_time_stats.geoparquet"

LAKES_PATH = Path("C:/work_dir/Lakes.gpkg")
LAKES_LAYER = "lakes_qgis"
SOIL_PATH = Path("C:/work_dir/Agro_Soil.gpkg")
SOIL_LAYER = None

EPSG_CODE = 2100  # Greek Grid (metric)
# False uses pixels whose centers fall inside each parcel. Set True for very small parcels.
ALL_TOUCHED = False

assert PARCELS_PATH.is_file(), f"Missing parcel file: {PARCELS_PATH}"

# Load the parcel dataset
parcel_geometry = read_data(str(PARCELS_PATH))
print(f"Loaded parcel dataset with {len(parcel_geometry)} features")
print(f"Original CRS: {parcel_geometry.crs}")

# Ensure the CRS is set correctly
if parcel_geometry.crs is None:
    print(f"Parcel dataset has no CRS; setting to EPSG:{EPSG_CODE}")
    parcel_geometry = parcel_geometry.set_crs(epsg=EPSG_CODE)
    write_data(parcel_geometry, str(PARCELS_PATH))
elif parcel_geometry.crs.to_epsg() != EPSG_CODE:
    print(f"Parcel CRS is {parcel_geometry.crs.to_epsg()}; converting to EPSG:{EPSG_CODE}")
    parcel_geometry = parcel_geometry.to_crs(epsg=EPSG_CODE)
    write_data(parcel_geometry, str(PARCELS_PATH))
else:
    print(f"Parcel dataset already uses EPSG:{EPSG_CODE}")

# Validate the CRS
validate_projected_metric_crs(parcel_geometry.crs, dataset_name="Parcel dataset")
ORIGINAL_PARCEL_CRS = parcel_geometry.crs
print(f"Parcel CRS that will be preserved: {ORIGINAL_PARCEL_CRS}")
parcel_geometry.head()

## Create or reuse the DEM mosaic

In [ ]:
def_files_path = "F:/esahub/Processed/data_common/dem_fl/ESA_10"
gdf_dem_tiles_path = os.path.join(def_files_path, "esa_tiles_dem_10m.gpkg")
gdf_dem_tiles = read_data(gdf_dem_tiles_path)
gdf_dem_tiles = gdf_dem_tiles.to_crs(epsg=EPSG_CODE)
tiles_to_copy = gdf_dem_tiles[gdf_dem_tiles.geometry.intersects(parcel_geometry.union_all())]
display(tiles_to_copy)
for _, tile in tiles_to_copy.iterrows():
    print(f"Processing tile: {tile}")
    src_tile_path = os.path.join(def_files_path, tile.file_name)
    dst_tile_path = os.path.join(DEM_DIR, tile.file_name)
    if not os.path.isfile(dst_tile_path):
        print(f"Copying {src_tile_path} to {dst_tile_path}")
        shutil.copy(src_tile_path, dst_tile_path)

dem_tiles = find_dem_tiles(DEM_DIR, MOSAIC_PATH)
# Reuses a valid existing mosaic. Set overwrite=True to rebuild it.
MOSAIC_PATH = create_dem_mosaic(dem_tiles, MOSAIC_PATH, overwrite=False)

## Add elevation and overwrite the parcel file

In [ ]:
if file_exists(str(MOSAIC_PATH)):
    parcels_with_elevation = append_parcel_elevation(
        parcels_path=PARCELS_PATH,
        mosaic_path=MOSAIC_PATH,
        output_path=PARCELS_PATH,
        all_touched=ALL_TOUCHED,
        overwrite=True,
    )
    assert parcels_with_elevation.crs == ORIGINAL_PARCEL_CRS

## Add nearest-lake features

Distance is the minimum polygon-to-polygon distance in EPSG:2100 metres. `nearest_lake_row_id` stores the stable GeoPackage feature ID (`fid`) of the matched lake.

In [ ]:
if file_exists(str(LAKES_PATH)):
    parcels_with_features = append_nearest_lake_features(
        parcels_path=PARCELS_PATH,
        lakes_path=LAKES_PATH,
        output_path=PARCELS_PATH,
        lakes_layer=LAKES_LAYER,
        overwrite=True,
    )
    assert parcels_with_features.crs == ORIGINAL_PARCEL_CRS

## Add the soil attribute

Each parcel receives `GAIOIKANOT` from the soil polygon intersecting its centroid. Centroids are calculated in the original projected parcel CRS.

In [ ]:
if file_exists(str(SOIL_PATH)):
    parcels_with_features = append_soil_feature(
        parcels_path=PARCELS_PATH,
        soil_path=SOIL_PATH,
        output_path=PARCELS_PATH,
        soil_layer=SOIL_LAYER,
        attribute_column="GAIOIKANOT",
        overwrite=True,
    )
    assert parcels_with_features.crs == ORIGINAL_PARCEL_CRS

## Checks

In [ ]:
feature_columns = [
    "elevation_mean_m",
    "elevation_valid_pixel_count",
    "distance_to_nearest_lake_m",
    "nearest_lake_row_id",
    "GAIOIKANOT",
]
saved_parcels = read_data(str(PARCELS_PATH))
assert saved_parcels.crs == ORIGINAL_PARCEL_CRS, (saved_parcels.crs, ORIGINAL_PARCEL_CRS)
validate_projected_metric_crs(saved_parcels.crs, dataset_name="Saved parcel dataset")
summary = saved_parcels[["elevation_mean_m", "elevation_valid_pixel_count", "distance_to_nearest_lake_m"]].describe()
missing_elevation = int(saved_parcels["elevation_mean_m"].isna().sum())
missing_lake = int(saved_parcels["distance_to_nearest_lake_m"].isna().sum())
missing_soil = int(saved_parcels["GAIOIKANOT"].isna().sum())
print(f"Rows: {len(saved_parcels):,}")
print(f"Preserved projected CRS: {saved_parcels.crs}")
print(f"Missing elevation: {missing_elevation:,}")
print(f"Missing nearest lake: {missing_lake:,}")
print(f"Missing soil attribute: {missing_soil:,}")
display(saved_parcels["GAIOIKANOT"].value_counts(dropna=False).head(10))
summary

In [ ]:
saved_parcels.head()